<a href="https://colab.research.google.com/github/1pawn0/GNNs-in-ComputerNetworks/blob/main/notebooks/caida_as_relationships_serial2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dataset:
[https://catalog.caida.org/dataset/as_relationships_serial_2](https://catalog.caida.org/dataset/as_relationships_serial_2)



In [ ]:
# @title Install dependencies based on torch version

import os
import torch

TORCH = ".".join(torch.__version__.split("+")[0].split(".")[:3])
CUDA = f"cu{torch.version.cuda.replace('.', '')[:3]}" if torch.version.cuda else "cpu"

os.environ["TORCH"] = TORCH
del TORCH
os.environ["CUDA"] = CUDA
del CUDA

%pip install -q pyg_lib torch_scatter torch_sparse torch_geometric -f https://data.pyg.org/whl/torch-${TORCH}+${CUDA}.html
%pip install -q "torch-geometric-temporal[index]" torch-geometric-signed-directed

import torch_geometric
import torch_geometric_temporal
import torch_geometric_signed_directed

print(torch.__version__)
print(torch_geometric.__version__)
print(torch_geometric_temporal.__version__)


In [ ]:
# @title Load data

from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
data_dir = Path("/content/drive/MyDrive/Research/datasets/caida_as_relationships_serial2_txt_files")
txt_data_filenames: list[str] = sorted(data_dir.walk().__next__()[2])
last_month_file: Path = data_dir / txt_data_filenames[-1]


In [ ]:
# @title Build a dataframe from raw txt file

import polars as pl

source_enum = pl.Enum(["bgp", "mlp"])
rel_enum = pl.Enum(["provider->customer", "peer-peer"])

df = pl.read_csv(
    last_month_file,
    has_header=False,
    separator="|",
    comment_prefix="#",
    new_columns=["as1", "as2", "rel", "source"],
    schema_overrides={
        "as1": pl.UInt32,
        "as2": pl.UInt32,
        "rel": pl.Int8,
        "source": pl.Utf8,
    },
).with_row_index()

df = df.with_columns(
    pl.col("source").str.split(",").cast(pl.List(source_enum)).alias("source"),
    pl
    .when(pl.col("rel") == -1)
    .then(pl.lit("provider->customer"))
    .when(pl.col("rel") == 0)
    .then(pl.lit("peer-peer"))
    .cast(rel_enum)
    .alias("rel_label"),
    (pl.col("rel") == -1).alias("is_p2c"),
    (pl.col("rel") == 0).alias("is_p2p"),
)

df = df.with_columns(
    pl.col("source").cast(pl.List(pl.Utf8)).list.contains("bgp").alias("from_bgp"),
    pl.col("source").cast(pl.List(pl.Utf8)).list.contains("mlp").alias("from_mlp"),
)
